In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, roc_curve

# Configuração do MLflow para criar o banco de dados local na pasta do notebook
mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("Previsao_Churn_Telco")

In [ ]:
def preparar_dados(caminho_csv):
    """
    Realiza a leitura do dataset, tratamento de nulos, encoding de variáveis 
    categóricas e o particionamento em conjuntos de treino e teste.
    """
    df = pd.read_csv(caminho_csv)
    
    # Tratamento da coluna TotalCharges: conversão de espaços em branco para NaN e preenchimento com 0
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)
    
    # Remoção da coluna de identificador único (não preditiva)
    if 'customerID' in df.columns:
        df = df.drop(columns=['customerID'])
        
    # Mapeamento da variável alvo
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
    
    # Separação das features (X) e do alvo (y)
    X = df.drop(columns=['Churn'])
    y = df['Churn']
    
    # Transformação das variáveis categóricas utilizando One-Hot Encoding
    X_encoded = pd.get_dummies(X, drop_first=True)
    
    # Particionamento dos dados (80% treino, 20% teste) preservando a proporção da classe alvo
    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X_encoded, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Normalização dos dados numéricos
    scaler = StandardScaler()
    X_treino_scaled = scaler.fit_transform(X_treino)
    X_teste_scaled = scaler.transform(X_teste)
    
    return X_treino_scaled, X_teste_scaled, y_treino, y_teste

In [ ]:
def train_all_models(models, X_train, X_test, y_train, y_test):
    """
    Itera sobre os modelos fornecidos, executa o ajuste (fit),
    avalia a métrica ROC-AUC, plota a Curva ROC e registra os artefatos no MLflow.
    """
    resultados = {}
    
    for nome_modelo, modelo in models.items():
        print(f"Executando pipeline para o modelo: {nome_modelo}...")
        
        # Inicializa o rastreamento do MLflow para o modelo iterado
        with mlflow.start_run(run_name=nome_modelo):
            
            # Ajuste do modelo
            modelo.fit(X_train, y_train)
            
            # Previsão das probabilidades para a classe positiva (Churn = 1)
            probabilidades = modelo.predict_proba(X_test)[:, 1]
            
            # Cálculo da métrica exigida
            roc_auc = roc_auc_score(y_test, probabilidades)
            resultados[nome_modelo] = roc_auc
            
            # Registro de parâmetros e métricas no MLflow
            mlflow.log_param("algoritmo", nome_modelo)
            mlflow.log_metric("roc_auc", roc_auc)
            mlflow.sklearn.log_model(modelo, f"modelo_{nome_modelo}")
            
            # Plotagem da Curva ROC
            fpr, tpr, _ = roc_curve(y_test, probabilidades)
            
            plt.figure(figsize=(6, 4))
            plt.plot(fpr, tpr, color='blue', label=f'AUC = {roc_auc:.4f}')
            plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Modelo Aleatório')
            plt.title(f'Curva ROC - {nome_modelo}')
            plt.xlabel('Taxa de Falsos Positivos (FPR)')
            plt.ylabel('Taxa de Verdadeiros Positivos (TPR)')
            plt.legend(loc='lower right')
            plt.grid(alpha=0.3)
            plt.show()
            
            print(f"Finalizado {nome_modelo}. ROC-AUC: {roc_auc:.4f}\n")
            
    return resultados

In [ ]:
def main():
    # Caminho do dataset (pasta data separada da pasta de notebooks)
    caminho_csv = os.path.join("..", "data", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
    
    # Verifica se o arquivo existe no caminho estipulado
    if not os.path.exists(caminho_csv):
        print(f"Erro: O arquivo não foi encontrado no caminho '{caminho_csv}'.")
        print("Certifique-se de que a pasta 'data' está um nível acima da pasta do notebook.")
        return
        
    print("Iniciando pré-processamento...")
    X_treino, X_teste, y_treino, y_teste = preparar_dados(caminho_csv)
    
    # Instanciação dos modelos clássicos obrigatórios
    modelos = {
        "Regressao_Logistica": LogisticRegression(max_iter=1000, random_state=42),
        "Arvore_de_Decisao": DecisionTreeClassifier(max_depth=5, random_state=42)
    }
    
    print("\nIniciando bateria de treinamentos...")
    resultados_finais = train_all_models(modelos, X_treino, X_teste, y_treino, y_teste)
    
    print("========================================")
    print("RESUMO DOS EXPERIMENTOS")
    print("========================================")
    for nome, auc in resultados_finais.items():
        print(f"{nome}: {auc:.4f}")
    print("========================================")
    print("Para acessar a interface do MLflow, execute no terminal:")
    print("mlflow ui --backend-store-uri sqlite:///mlruns.db")

if __name__ == "__main__":
    main()